In [ ]:
!pip uninstall -y segmentation-models-pytorch
!pip install -U segmentation-models-pytorch

print("\n\n--- UPGRADE COMPLETE. FORCING KERNEL RESTART. ---")
print("Please wait for the kernel to restart, then re-run the main script cell.")

# This line will force the kernel to restart
import os
os._exit(0)

Found existing installation: segmentation-models-pytorch 0.5.0
Uninstalling segmentation-models-pytorch-0.5.0:
  Successfully uninstalled segmentation-models-pytorch-0.5.0


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import cv2
import numpy as np
import os
import glob
from tqdm import tqdm
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- SOTA 1 UPGRADE (IMPORTS) ---
from segmentation_models_pytorch.losses import DiceLoss
import segmentation_models_pytorch.tta as tta  # <-- This will now work
# ---------------------------------

# --- SOTA 1 UPGRADE (MODEL) ---
def get_model(device):
    model = smp.Unet(
        encoder_name="efficientnet-b4",  # <-- Upgraded from mobilenet_v2
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    ).to(device)
    
    return model

# --- SOTA 1 UPGRADE (LOSS) ---
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        """
        Combines BCEWithLogitsLoss and DiceLoss.
        alpha: weight for BCE, (1-alpha) for Dice
        """
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss(mode='binary')
        
    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        dice_loss = self.dice(inputs, targets)
        return self.alpha * bce_loss + (1 - self.alpha) * dice_loss
# -------------------------------

class ProjectSegmentationDataset(Dataset):
    def __init__(self, imageDir, maskDir, image_paths, transform=None, is_test=False):
        super().__init__()
        self.imageDir = imageDir
        self.maskDir = maskDir
        self.image_paths = image_paths
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        imgPath = self.image_paths[index]
        
        img = cv2.imread(imgPath, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Could not read image: {imgPath}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.is_test:
            img_h, img_w = img.shape[:2]
            if self.transform:
                augmented = self.transform(image=img)
                img = augmented['image']
            return img, os.path.basename(imgPath), (img_h, img_w)

        filename = os.path.basename(imgPath)
        filename_stem = os.path.splitext(filename)[0]
        maskFilename = f"{filename_stem}_mask.png"
        maskPath = os.path.join(self.maskDir, maskFilename)
        mask = cv2.imread(maskPath, cv2.IMREAD_GRAYSCALE)
        if mask is None:
                raise RuntimeError(f"Could not read mask for path: {maskPath}")
        mask = (mask > 127).astype("float32")
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        mask = mask.unsqueeze(0)
        return img, mask

def get_transforms(is_train, img_size=(512, 512)):
    if is_train:
        return A.Compose([
            A.Resize(img_size[0], img_size[1]),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=30, p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size[0], img_size[1]),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

def get_inference_transform():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def saveCheckpoint(state, filename="checkpoint.pth.tar"):
    print("=> Saving new best model")
    torch.save(state, filename)

def calculateIou(preds, masks, threshold=0.5):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()
    masks = (masks > 0.5).float()
    intersection = (preds * masks).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean()

def evaluateModel(loader, model, lossFn, device):
    model.eval()
    totalLoss, totalIou = 0.0, 0.0
    
    progressBar = tqdm(loader, desc="[Validation]")
    
    with torch.no_grad():
        for images, masks in progressBar:
            images, masks = images.to(device), masks.to(device)
            predictions = model(images)
            loss = lossFn(predictions, masks)
            iou = calculateIou(predictions, masks)
            totalLoss += loss.item()
            totalIou += iou.item()
            progressBar.set_postfix(loss=f"{loss.item():.4f}", iou=f"{iou.item():.4f}")
            
    model.train()
    return {"loss": totalLoss / len(loader), "iou": totalIou / len(loader)}

def trainAndValidateModel(model, modelName, trainLoader, valLoader, config, device):
    
    lossFn = CombinedLoss().to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
    
    print(f"\n===== Starting Training for: {modelName} on {device} =====")
    bestValIou = 0.0
    checkpointFilename = f"best_{modelName.replace(' ', '_')}.pth.tar"
    
    for epoch in range(config["epochs"]):
        model.train()
        trainLoss, trainIou = 0.0, 0.0
        
        progressBar = tqdm(trainLoader, desc=f"Epoch {epoch+1}/{config['epochs']} [Training]")
        
        for images, masks in progressBar:
            images, masks = images.to(device), masks.to(device)
            
            predictions = model(images)
            loss = lossFn(predictions, masks)
            iou = calculateIou(predictions, masks)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            trainLoss += loss.item()
            trainIou += iou.item()
            progressBar.set_postfix(loss=f"{loss.item():.4f}", iou=f"{iou.item():.4f}")

        avgTrainLoss = trainLoss / len(trainLoader)
        avgTrainIou = trainIou / len(trainLoader)
        
        valMetrics = evaluateModel(valLoader, model, lossFn, device)

        print(f"--- Epoch {epoch+1} Summary ---")
        print(f"Training -> Loss: {avgTrainLoss:.4f}, IoU: {avgTrainIou:.4f}")
        print(f"Validation -> Loss: {valMetrics['loss']:.4f}, IoU: {valMetrics['iou']:.4f}")

        if valMetrics["iou"] > bestValIou:
            bestValIou = valMetrics["iou"]
            checkpoint = {
                "stateDict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "bestValIou": bestValIou
                }
            saveCheckpoint(checkpoint, filename=checkpointFilename)

    print(f"--- Training complete for {modelName}. Best model saved as '{checkpointFilename}' ---")
    return checkpointFilename

def generate_predictions(model, loader, device, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    model.eval() 
    print(f"Generating predictions and saving to {output_dir}...")
    
    with torch.no_grad():
        for images, filenames, original_sizes in tqdm(loader, desc="Predicting masks"):
            images = images.to(device)
            
            predictions = model(images)
            preds_binary = (torch.sigmoid(predictions) > 0.5).float()
            
            for i in range(preds_binary.shape[0]):
                mask_tensor = preds_binary[i].cpu()
                
                orig_h, orig_w = original_sizes[0][i].item(), original_sizes[1][i].item()

                mask_resized = F.interpolate(
                    mask_tensor.unsqueeze(0), 
                    size=(orig_h, orig_w), 
                    mode='nearest'
                ).squeeze() 
                
                mask = mask_resized.numpy()
                mask_to_save = mask.astype(np.uint8) 
                
                filename_stem = os.path.splitext(filenames[i])[0]
                output_filename = f"{filename_stem}_mask.png" 
                output_path = os.path.join(output_dir, output_filename)
                
                cv2.imwrite(output_path, mask_to_save * 255) 


#main loop
all_image_files = sorted(glob.glob(os.path.join("./data/images/", "*.jpg"))) 
train_val_paths = []
test_paths = []


if __name__ == '__main__':
    
    CONFIG = {
        "learning_rate": 1e-4,
        "epochs": 25, 
        "batch_size": 1,
    }
    
    IMAGE_DIR = "./data/images/"
    MASK_DIR = "./data/masks/"
    PREDICTION_DIR = "./data/masks/" 
    
    MODEL_NAME = "UNet-EfficientNetB4"
    BEST_MODEL_PATH = f"best_{MODEL_NAME.replace(' ', '_')}.pth.tar"
    TRAIN_IMG_SIZE = (512, 512) 
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {DEVICE}")
    
    print(f"Scanning {len(all_image_files)} images to find pairs...")
    for img_path in tqdm(all_image_files, desc="Scanning files"):
        filename_stem = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(MASK_DIR, f"{filename_stem}_mask.png")
        
        if os.path.exists(mask_path):
            train_val_paths.append(img_path)
        else:
            test_paths.append(img_path)
            
    print(f"Found {len(train_val_paths)} labeled images (for training/validation).")
    print(f"Found {len(test_paths)} unlabeled images (for testing).")
    
    if not os.path.exists(BEST_MODEL_PATH):
        print(f"\n--- Model file not found. STARTING TRAINING PHASE ---")
        
        total_labeled = len(train_val_paths)
        val_size = max(1, total_labeled // 10) # Ensure val_size is at least 1
        train_size = total_labeled - val_size
        
        if total_labeled == 0:
            print("Error: No labeled data found. Cannot train.")
        elif train_size == 0:
            print("Warning: Not enough data for a train/val split. Using all for training.")
            train_img_paths = train_val_paths
            val_img_paths = []
        else:
            train_img_paths, val_img_paths = random_split(train_val_paths, [train_size, val_size])
        
        if train_size > 0:
            train_dataset = ProjectSegmentationDataset(
                IMAGE_DIR, MASK_DIR, train_img_paths, 
                transform=get_transforms(is_train=True, img_size=TRAIN_IMG_SIZE)
            )
            train_loader = DataLoader(
                dataset=train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
                num_workers=0, pin_memory=True
            )
        else:
            train_loader = None

        if val_size > 0:
            val_dataset = ProjectSegmentationDataset(
                IMAGE_DIR, MASK_DIR, val_img_paths, 
                transform=get_transforms(is_train=False, img_size=TRAIN_IMG_SIZE)
            )
            val_loader = DataLoader(
                dataset=val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
                num_workers=0, pin_memory=True
            )
        else:
            val_loader = None
            
        if train_loader is not None and val_loader is not None:
            model = get_model(DEVICE)
            
            BEST_MODEL_PATH = trainAndValidateModel(
                model=model, modelName=MODEL_NAME, trainLoader=train_loader,
                valLoader=val_loader, config=CONFIG, device=DEVICE
            )
            
            print(f"\nTraining finished. Best model saved to: {BEST_MODEL_PATH}")
        else:
            print("Skipping training due to insufficient data for train/val loaders.")
            
    else:
        print(f"\n--- Found existing model at {BEST_MODEL_PATH}. SKIPPING TRAINING. ---")

    print("\n--- STARTING MASK PHASE ---")
    
    prediction_model = get_model(DEVICE)
    try:
        checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
        prediction_model.load_state_dict(checkpoint['stateDict'])
        print(f"Loaded best model weights from {BEST_MODEL_PATH}")
    except FileNotFoundError:
        print(f"Error: Could not find model file at {BEST_MODEL_PATH}. Cannot run predictions.")
        pass 
    except Exception as e:
        print(f"Error loading model: {e}. Check if BEST_MODEL_PATH is correct.")
        pass

    if 'checkpoint' in locals() and len(test_paths) > 0:
        
        print("Wrapping model in TTA (Test-Time Augmentation) wrapper...")
        tta_model = tta.TTAWrapper(
            prediction_model, 
            tta.aliases.d4_transform(), 
            merge_mode='mean'
        )
        tta_model.to(DEVICE)

        test_dataset_for_gen = ProjectSegmentationDataset(
            IMAGE_DIR, MASK_DIR, test_paths, 
            transform=get_transforms(is_train=False, img_size=TRAIN_IMG_SIZE), 
            is_test=True
        )
        
        test_loader = DataLoader(
            dataset=test_dataset_for_gen, batch_size=1,
            shuffle=False, num_workers=0, pin_memory=True
        )
        
        generate_predictions(tta_model, test_loader, DEVICE, PREDICTION_DIR)
        
        print("--- Segmentation Pipeline Complete ---")
    elif len(test_paths) == 0:
        print("No test images found to predict on.")
    else:
        print("Skipping prediction phase as model was not loaded.")

ModuleNotFoundError: No module named 'segmentation_models_pytorch.tta'